# Activity 5: Building the dataset

We put Trinidad and Tobago's quarterly real GDP together with **your own Google Trends
terms** from Activity 3, the night-time lights and the oil price:

| Source | File | Frequency |
|---|---|---|
| Real GDP (CBTT, constant prices, provided) | `data/gdp.csv` | quarterly |
| Google Trends (**your** terms, from Activity 3) | `raw/gtrends.csv` | monthly |
| Night-time lights, country mean (provided) | `data/ntl_shape.csv` | daily |
| Brent oil price (EIA) | `data/eia_brent.csv` | daily |

Steps: load → clean → seasonally adjust **each series at its own frequency** → merge to
quarterly → growth rates → a first OLS. Then the question that matters: does the
relationship survive without 2020?

Kernel: `nwcst`. Cells marked `# YOUR CODE HERE` are yours.

In [ ]:
# ===========================================================================
# CONFIG LOADER: edit config.py, NOT this cell.
# Gives you `cfg` (every setting) and `P` (the standard folders).
# ===========================================================================
import os, sys, importlib.util

# config.py lives in activity/. From activity/ that is the current folder;
# from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))
sys.path.insert(0, CODE_DIR)       # so `import nwcst_helpers` works from sols/ too

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)     # raw/, data/, output/ under public/

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE})  data -> {P['data']}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

## 1. GDP

`gdp.csv` stamps each quarter on its **last** month (2026Q1 = `2026-03-01`).

In [ ]:
gdp = pd.read_csv(os.path.join(P["data"], cfg.GDP_CSV), index_col="date", parse_dates=True)
gdp = gdp.rename(columns={"rgdp": "RGDP0000"})
print(gdp.index.min().date(), "->", gdp.index.max().date(), f"({len(gdp)} quarters)")
gdp.tail()

In [ ]:
ax = gdp["RGDP0000"].plot(figsize=(9, 3), title="Trinidad and Tobago: real GDP (TT$ mn, constant prices)")
ax.set_xlabel("");

GDP is **not** seasonally adjusted. With quarterly data one seasonal cycle is **4**
observations, so `period=4`.

In [ ]:
seasonal_decompose(gdp["RGDP0000"], model="additive", extrapolate_trend="freq", period=4).plot();

In [ ]:
gdp["RGDP0000"] = gdp["RGDP0000"] - seasonal_decompose(
    gdp["RGDP0000"], model="additive", extrapolate_trend="freq", period=4).seasonal

## 2. Google Trends

The terms **you** downloaded in Activity 3, saved in `raw/gtrends.csv`: one column per term,
one row per month. Months with too few searches to report are empty.

In [ ]:
gt = pd.read_csv(os.path.join(P["raw"], "gtrends.csv"), index_col="date", parse_dates=True)
gt = gt.loc["2012":]                      # same window as the night lights
gt.tail()

Average your terms into one index, one value per month:

In [ ]:
gt["gt_mean"] = gt.mean(axis=1)
gt.tail()

### Selecting columns: `df[ ]` or `df[[ ]]`?

**Single brackets** give a **Series**: one column, shown as a list of values with the name at the bottom.

In [ ]:
gt["gt_mean"].tail()

**Double brackets** give a **DataFrame**: a table, even with a single column. The inner brackets are a
*list* of column names, so you could ask for several at once (`gt[["gt_mean", "gt_amazon_tt"]]`).

In [ ]:
gt[["gt_mean"]].tail()

We keep only the index, as a DataFrame:

In [ ]:
gt = gt[["gt_mean"]]
gt.tail()

In [ ]:
gt.plot(figsize=(9, 3), title="Google Trends: average of your terms");

**Exercise 1.** Seasonally adjust `gt_mean` (monthly, so `period=12`) and plot the raw and adjusted series together.

In [ ]:
# YOUR CODE HERE: create gt["gt_mean_sa"] = gt_mean minus its seasonal component, then plot both

In [ ]:
gt = gt[["gt_mean_sa"]].rename(columns={"gt_mean_sa": "gt_mean"})

## 3. Night-time lights

The country-mean radiance (provided; Activity 4 shows how it is built) is **daily**. Cloudy nights leave gaps, so we
average to months first.

In [ ]:
ntl = pd.read_csv(os.path.join(P["data"], cfg.NTL_CSV), index_col="date", parse_dates=True)
ntl = ntl[["shape_mean"]].resample("MS").mean().rename(columns={"shape_mean": "ntl_mean"})
ntl = ntl.loc[:"2026-09-01"]                     # the current month is incomplete
ntl.plot(figsize=(9, 3), title="Night-time lights: monthly mean radiance (nW/cm²/sr)");

**Exercise 2.** Seasonally adjust `ntl_mean` exactly as you did for Google Trends.

In [ ]:
# YOUR CODE HERE: ntl["ntl_mean_sa"] = ...

In [ ]:
ntl = ntl[["ntl_mean_sa"]].rename(columns={"ntl_mean_sa": "ntl_mean"})

## 4. Brent oil price

Trinidad and Tobago is an energy exporter, so the oil price is a natural candidate. The EIA
series in `data/eia_brent.csv` is **daily**; we average it to months. Prices are not
seasonally adjusted.

In [ ]:
brent = pd.read_csv(os.path.join(P["data"], "eia_brent.csv"), index_col="date", parse_dates=True)
brent = brent.rename(columns={"brent price": "brent"}).resample("MS").mean().loc["2012":]
brent.plot(figsize=(9, 3), title="Brent, US$ per barrel (monthly mean)");

## 5. Merge to quarterly

Put the four series side by side, as they are:

In [ ]:
data = pd.concat([gdp, gt, ntl, brent], axis=1).sort_index()
data.loc["2025"]

Something is wrong. GDP has a value only **once every three months** (on the last month of each
quarter), while the other three have one every month. A regression needs **one row per quarter,
with every column filled**. Try dropping the incomplete rows:

In [ ]:
data.dropna().loc["2025"]

Only March, June, September and December survive, and for those rows we kept a **single month**
of each monthly series, throwing the other two away. What we want next to each quarter's GDP is
the **quarterly average** of the monthly series.

**Exercise 3.** Resample `data` to quarters (`"QS"`, quarter start) using the mean. Each quarter gets
the average of its three months, and GDP, with one value per quarter, falls into its own quarter.

In [ ]:
data = <YOUR CODE HERE: resample data to quarters ("QS") with the mean>
data.loc["2025"]

In [ ]:
data = data.loc[:data["RGDP0000"].last_valid_index()]   # stop at the last published GDP quarter
data.tail(8)

## 6. Growth rates

Quarter-on-quarter growth, in percent.

In [ ]:
growth = data.pct_change(fill_method=None).mul(100)
growth.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for col, color in [("RGDP0000", "#003865"), ("gt_mean", "#009CA7"), ("ntl_mean", "#F5A623")]:
    ax.plot(growth[col], label=col, color=color, lw=2 if col == "RGDP0000" else 1.2)
ax.axhline(0, color="black", lw=0.5)
ax.set_title("Quarterly growth, % q/q (seasonally adjusted)")
ax.legend(frameon=False);

In [ ]:
growth.corr().round(2)

## 7. A first OLS

$$\text{GDP growth}_t = \beta_0 + \beta_1\,\text{Google Trends}_t + \beta_2\,\text{NTL}_t + \varepsilon_t$$

In [ ]:
import statsmodels.api as sm

d1 = growth[["RGDP0000", "gt_mean", "ntl_mean"]].dropna()
m1 = sm.OLS(d1["RGDP0000"], sm.add_constant(d1[["gt_mean", "ntl_mean"]])).fit()
print(f"{len(d1)} quarters, {d1.index.min().date()} -> {d1.index.max().date()}")
print(m1.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(d1.index, d1["RGDP0000"], "o-", color="#003865", label="actual")
ax.plot(d1.index, m1.fittedvalues, "--", color="#F5A623", label="fitted")
ax.axhline(0, color="black", lw=0.5)
ax.set_title(f"GDP growth, actual vs fitted (R² = {m1.rsquared:.2f})")
ax.legend(frameon=False);

## 8. The question: COVID influence 

COVID-19 produced the largest quarterly swings in the sample. A regression can look
good simply because two or three quarters line up.

**Exercise 4.** Re-estimate the model without 2020 and compare the coefficients and R².

In [ ]:
# YOUR CODE HERE: d2 = d1[d1.index.year != 2020]; m2 = sm.OLS(...).fit()

**Exercise 5.** Add `brent` to the regression (with and without 2020). Does the oil price
help more than the searches and the lights in an energy-heavy economy? Why might the
*same-quarter* oil price be a weak predictor of real (constant-price) GDP?

In [ ]:
# YOUR CODE HERE